# Ventana de 365 dias: rearresto, reimputacion y caso abierto

Todo se calcula sobre la base ajustada del script anterior, sin duraciones
negativas y sin el 1% superior de cada variable. Si esa base ya esta
guardada se lee; si no, se reconstruye con la misma logica.

La limpieza del script anterior es por radicado y no se toca. Las tres
variables de seguimiento son de la persona, asi que la base se expande a
una fila por IDENTIFICACION + radicado antes de armar la ventana.

Equivalencias de columnas: dummy_boleta_salida = BoletaSalida,
fecha_boleta_salida = FechaBoletaSalida, fecha_legalizacion =
FechaLegalizacion, fecha_medida_aseguramiento = FechaMedida,
fecha_imputacion_cargos = FechaImputacion.

La base no trae fecha de radicacion, asi que para caso abierto se usa
FECHA (la captura, presente en el 100% de los casos) con respaldo en
FechaLegalizacion y FechaImputacion. Queda guardada en
fecha_evento_radicado por si hay que auditarla.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DESCARGAS = Path.home() / "Downloads"

RUTA_BASE = DESCARGAS / "base_final.parquet"
RUTA_CAPTURAS = DESCARGAS / "capturas_with_status.parquet"
RUTA_BASE_MUESTRA = DESCARGAS / "muestra_revision.csv"
RUTA_AJUSTADA = DESCARGAS / "base_ajustada.parquet"
RUTA_SALIDA = DESCARGAS / "base_con_seguimiento.parquet"

COL_PROCESO = "radicado"
COL_ID = "IDENTIFICACION"

COL = {
    "captura": "FECHA",
    "legalizacion": "FechaLegalizacion",
    "imputacion": "FechaImputacion",
    "medida": "FechaMedida",
    "acusacion": "FechaAcusacion",
    "sentencia": "FechaSentencia",
    "boleta_salida": "FechaBoletaSalida",
}

COL_DUMMY_BOLETA = "BoletaSalida"

COL_LLAVE = "llave_caso"

DESCARTAR_NEGATIVOS = True
PERCENTIL_CORTE = 0.99

VENTANA_DIAS = 365

In [ ]:
from itertools import permutations

import numpy as np
import pandas as pd

ANIO_MINIMO = 2004
ANIO_MAXIMO = 2026

CADENAS = [
    ["FECHA", "FechaLegalizacion", "FechaImputacion", "FechaMedida",
     "FechaAcusacion", "FechaSentencia"],
    ["FECHA", "FechaBoletaDetencion", "FechaBoletaSalida"],
]

COL_PROCESO_FECHAS = "radicado"
MAXIMO_ANIOS_PROCESO = 10
HOLGURA_RADICADO = 15

COLUMNAS_FECHA = list(dict.fromkeys(c for cadena in CADENAS for c in cadena))


def _a_datetime_ancho(serie: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie.astype("datetime64[us]")
    convertida = pd.to_datetime(serie, errors="coerce")
    if pd.api.types.is_datetime64_any_dtype(convertida):
        return convertida.astype("datetime64[us]")
    return convertida


def candidatos_anio(anio, anio_min=ANIO_MINIMO, anio_max=ANIO_MAXIMO):
    digitos = str(int(anio))
    if len(digitos) != 4:
        return []

    posibles = set()

    for p in permutations(digitos):
        if p[0] != "0":
            posibles.add(int("".join(p)))

    for i in range(4):
        for d in "0123456789":
            if d == digitos[i]:
                continue
            nuevo = digitos[:i] + d + digitos[i + 1:]
            if nuevo[0] != "0":
                posibles.add(int(nuevo))

    return sorted(a for a in posibles if anio_min <= a <= anio_max and a != int(anio))


def _cambiar_anio(fecha, anio):
    try:
        return pd.Timestamp(year=anio, month=fecha.month, day=fecha.day)
    except ValueError:
        return pd.NaT


def _vecinos(columna, cadenas, presentes):
    antes, despues = set(), set()
    for cadena in cadenas:
        cadena = [c for c in cadena if c in presentes]
        if columna not in cadena:
            continue
        i = cadena.index(columna)
        antes.update(cadena[:i])
        despues.update(cadena[i + 1:])
    return sorted(antes), sorted(despues)


def diagnosticar_fechas(df, columnas=None, anio_min=ANIO_MINIMO,
                        anio_max=ANIO_MAXIMO):
    columnas = COLUMNAS_FECHA if columnas is None else list(columnas)
    piso = pd.Timestamp(year=anio_min, month=1, day=1)
    techo = pd.Timestamp(year=anio_max, month=12, day=31)
    hallazgos = []

    for columna in columnas:
        if columna not in df.columns:
            continue
        serie = _a_datetime_ancho(df[columna])
        if not pd.api.types.is_datetime64_any_dtype(serie):
            continue

        fuera = serie.notna() & ((serie < piso) | (serie > techo))
        if not fuera.any():
            continue

        valores = serie[fuera]
        anios = valores.dt.year
        for anio, cuantos in anios.value_counts().items():
            candidatos = candidatos_anio(int(anio), anio_min, anio_max)
            hallazgos.append({
                "columna": columna,
                "anio": int(anio),
                "casos": int(cuantos),
                "ejemplo": valores[anios == anio].iloc[0].strftime("%Y-%m-%d"),
                "candidatos": ", ".join(str(a) for a in candidatos[:5]),
            })

    if not hallazgos:
        print(f"Ninguna fecha fuera del rango {anio_min}-{anio_max}")
        return pd.DataFrame()

    return pd.DataFrame(hallazgos).sort_values("casos", ascending=False)


def deducir_orden(df, columnas=None, ancla="FECHA"):
    columnas = COLUMNAS_FECHA if columnas is None else list(columnas)
    presentes = [c for c in columnas if c in df.columns]
    referencia = _a_datetime_ancho(df[ancla])
    filas = []
    for c in presentes:
        dias = (_a_datetime_ancho(df[c]) - referencia).dt.days
        dias = dias[dias.notna() & dias.between(-3650, 3650)]
        filas.append({"columna": c, "casos": len(dias),
                      "mediana_dias": dias.median(),
                      "p25": dias.quantile(0.25), "p75": dias.quantile(0.75)})
    return pd.DataFrame(filas).sort_values("mediana_dias")


def reparar_fechas(df, columna, cadenas=CADENAS, col_proceso=COL_PROCESO_FECHAS,
                   anio_min=ANIO_MINIMO, anio_max=ANIO_MAXIMO,
                   max_anios=MAXIMO_ANIOS_PROCESO, usar_radicado=True,
                   holgura_radicado=HOLGURA_RADICADO, aplicar=True, verbose=True):
    if columna not in df.columns:
        return df, pd.DataFrame()

    antes, despues = _vecinos(columna, cadenas, set(df.columns))
    if not antes and not despues:
        return df, pd.DataFrame()

    piso_rango = pd.Timestamp(year=anio_min, month=1, day=1)
    techo_rango = pd.Timestamp(year=anio_max, month=12, day=31)

    serie = _a_datetime_ancho(df[columna])
    malas = serie.notna() & ((serie < piso_rango) | (serie > techo_rango))
    if not malas.any():
        return df, pd.DataFrame()

    validas = {}
    for c in set(antes) | set(despues):
        s = _a_datetime_ancho(df[c])
        validas[c] = s.where(s.notna() & s.ge(piso_rango) & s.le(techo_rango))

    por_radicado = {}
    if usar_radicado and col_proceso in df.columns:
        auxiliar = pd.DataFrame(validas)
        auxiliar[col_proceso] = df[col_proceso].to_numpy()
        agrupado = auxiliar.groupby(col_proceso)
        por_radicado["min"] = agrupado.min()
        por_radicado["max"] = agrupado.max()

    registros = []

    for fila in df.index[malas]:
        original = serie.loc[fila]
        proceso = df.at[fila, col_proceso] if por_radicado else None

        def ancla(columna_ancla, extremo):
            propio = validas[columna_ancla].loc[fila]
            if pd.notna(propio):
                return propio, "fila"
            if por_radicado and proceso in por_radicado[extremo].index:
                v = por_radicado[extremo].at[proceso, columna_ancla]
                if pd.notna(v):
                    return v, "radicado"
            return pd.NaT, None

        holgura = pd.Timedelta(days=holgura_radicado)
        origen = set()

        anteriores = []
        for c in antes:
            v, fuente = ancla(c, "min")
            if pd.notna(v):
                anteriores.append(v - holgura if fuente == "radicado" else v)
                origen.add(fuente)
        piso = max(anteriores) if anteriores else piso_rango

        posteriores = []
        for c in despues:
            v, fuente = ancla(c, "max")
            if pd.notna(v):
                posteriores.append(v + holgura if fuente == "radicado" else v)
                origen.add(fuente)
        techo = (min(posteriores) if posteriores
                 else min(piso + pd.DateOffset(years=max_anios), techo_rango))

        sobreviven = [p for p in (_cambiar_anio(original, a)
                                  for a in candidatos_anio(original.year,
                                                           anio_min, anio_max))
                      if pd.notna(p) and piso <= p <= techo]

        registros.append({
            "fila": fila, "columna": columna, "original": original,
            "ancla_previa": piso if anteriores else pd.NaT, "techo": techo,
            "origen_ancla": ("fila" if origen == {"fila"} else
                             "radicado" if origen == {"radicado"} else
                             "mixta" if origen else "sin ancla"),
            "candidatos": len(sobreviven),
            "reparada": sobreviven[0] if len(sobreviven) == 1 else pd.NaT,
            "decision": ("reparada" if len(sobreviven) == 1
                         else "sin candidato" if not sobreviven else "ambigua"),
        })

    informe = pd.DataFrame(registros)

    if verbose:
        conteo = informe["decision"].value_counts()
        print(f"  {columna}: {len(informe)} imposibles -> "
              + ", ".join(f"{v} {k}" for k, v in conteo.items())
              + ("" if aplicar else "   (simulación)"))

    if not aplicar:
        return df, informe

    arregladas = informe[informe["decision"] == "reparada"]
    serie = serie.mask(malas)
    if len(arregladas):
        serie.loc[arregladas["fila"]] = arregladas["reparada"].to_numpy()
    df[columna] = serie.astype("datetime64[ns]")
    return df, informe


REGISTRO_FECHAS = {}


def fechas_a_datetime(df, columnas=None, anio_min=ANIO_MINIMO,
                      anio_max=ANIO_MAXIMO, reportar=True):
    piso = pd.Timestamp(year=anio_min, month=1, day=1)
    techo = pd.Timestamp(year=anio_max, month=12, day=31)
    columnas = list(COL.values()) if columnas is None else list(columnas)
    resumen = []

    for columna in columnas:
        if columna not in df.columns:
            continue

        nulos_originales = int(df[columna].isna().sum())
        serie = _a_datetime_ancho(df[columna])

        if not pd.api.types.is_datetime64_any_dtype(serie):
            resumen.append({"columna": columna, "estado": "no se pudo leer",
                            "ilegibles": np.nan, "fuera_de_rango": np.nan})
            continue

        ilegibles = int(serie.isna().sum() - nulos_originales)
        fuera = serie.notna() & ((serie < piso) | (serie > techo))
        n_fuera = int(fuera.sum())

        if n_fuera:
            REGISTRO_FECHAS[columna] = df.loc[fuera, [columna]].copy()
            serie = serie.mask(fuera)

        df[columna] = serie.astype("datetime64[ns]")
        resumen.append({"columna": columna, "estado": "ok",
                        "ilegibles": ilegibles, "fuera_de_rango": n_fuera})

    if reportar and resumen:
        tabla = pd.DataFrame(resumen)
        total = int(tabla["fuera_de_rango"].fillna(0).sum())
        if total:
            print(f"  anuladas por seguir fuera de {anio_min}-{anio_max}: {total:,}")
            print(tabla[tabla["fuera_de_rango"].fillna(0) > 0].to_string(index=False))
        else:
            print(f"  todas las fechas dentro de {anio_min}-{anio_max}")

    return df


print("=" * 70)
print("DIAGNÓSTICO")
print("=" * 70)

diagnostico = diagnosticar_fechas(data)
if len(diagnostico):
    print(f"Años imposibles distintos: {len(diagnostico)}   "
          f"filas afectadas: {diagnostico['casos'].sum():,}\n")
    print(diagnostico.head(20).to_string(index=False))


print("\n" + "=" * 70)
print("PASO 1. REPARACIÓN")
print("=" * 70)

informes = []
for columna in COLUMNAS_FECHA:
    if columna not in data.columns:
        continue
    data, informe = reparar_fechas(data, columna, aplicar=True)
    if len(informe):
        informes.append(informe)


if informes:
    informe_fechas = pd.concat(informes, ignore_index=True)
    informe_fechas.to_csv(DESCARGAS / "informe_fechas.csv", index=False)
    print("\nDecisiones:")
    print(informe_fechas["decision"].value_counts().to_string())
    print("\nOrigen del ancla:")
    print(informe_fechas["origen_ancla"].value_counts().to_string())
    print(f"\nInforme guardado en {DESCARGAS / 'informe_fechas.csv'}")
else:
    informe_fechas = pd.DataFrame()
    print("  no hubo fechas imposibles que reparar")


print("\n" + "=" * 70)
print("PASO 2. ANULAR LO QUE NO SE PUDO REPARAR")
print("=" * 70)

data = fechas_a_datetime(data, columnas=set(COL.values()) | set(COLUMNAS_FECHA))

print("\n" + "=" * 70)
print("PASO 3. VERIFICACIÓN")
print("=" * 70)

revisables = [c for c in set(COL.values()) | set(COLUMNAS_FECHA)
              if c in data.columns]


fuera = sum(int((data[c].notna()
                 & ~data[c].dt.year.between(ANIO_MINIMO, ANIO_MAXIMO)).sum())
            for c in revisables)
print(f"  fechas fuera de rango que quedaron: {fuera}")

no_ns = [c for c in revisables if str(data[c].dtype) != "datetime64[ns]"]
print(f"  columnas que no quedaron en datetime64[ns]: {no_ns if no_ns else 'ninguna'}")

violaciones = []
for cadena in CADENAS:
    presentes = [c for c in cadena if c in data.columns]
    for previo, posterior in zip(presentes, presentes[1:]):
        par = data[[previo, posterior]].dropna()
        if len(par):
            violaciones.append({
                "par": f"{previo} -> {posterior}",
                "invertidos": int((par[posterior] < par[previo]).sum()),
                "de": len(par),
            })


if violaciones:
    tabla = pd.DataFrame(violaciones)
    tabla["%"] = (100 * tabla["invertidos"] / tabla["de"]).round(2)
    print("\n  pares en orden invertido:")
    print(tabla.to_string(index=False))

## Pasos 1 a 3. Base ajustada

Duracion cruda, se excluyen las negativas, se calcula el p99 sobre lo que
queda y se excluye lo que este por encima. Un umbral por variable, sobre
la base total.

In [ ]:
INTERVALOS = [
    ("dias_captura_legalizacion",       "captura",      "legalizacion"),
    ("dias_legalizacion_imputacion",    "legalizacion", "imputacion"),
    ("dias_imputacion_medida",          "imputacion",   "medida"),
    ("dias_medida_acusacion",           "medida",       "acusacion"),
    ("dias_acusacion_sentencia",        "acusacion",    "sentencia"),
    ("dias_captura_sentencia",          "captura",      "sentencia"),
    ("dias_legalizacion_sentencia",     "legalizacion", "sentencia"),
    ("dias_captura_boleta_salida",      "captura",      "boleta_salida"),
    ("dias_legalizacion_boleta_salida", "legalizacion", "boleta_salida"),
]

VARIABLES = [nombre for nombre, _, _ in INTERVALOS]


def fechas_a_datetime(df: pd.DataFrame) -> pd.DataFrame:
    for columna in COL.values():
        if columna in df.columns:
            df[columna] = pd.to_datetime(df[columna], errors="coerce").astype(
                "datetime64[ns]"
            )
    return df


if RUTA_BASE.exists():
    data = pd.read_parquet(RUTA_BASE)
    origen = RUTA_BASE.name
else:
    data = pd.read_csv(RUTA_BASE_MUESTRA, encoding="utf-8-sig")
    origen = RUTA_BASE_MUESTRA.name


print(f"Base original: {origen} ({len(data):,} filas en formato largo)")

faltantes = [c for c in list(COL.values()) + [COL_PROCESO, COL_ID, COL_DUMMY_BOLETA]
             if c not in data.columns]
if faltantes:
    raise KeyError(f"Faltan columnas en la base: {faltantes}")


data = fechas_a_datetime(data)
data[COL_PROCESO] = data[COL_PROCESO].astype("string")

if RUTA_AJUSTADA.exists():
    proc = fechas_a_datetime(pd.read_parquet(RUTA_AJUSTADA))
    proc[COL_PROCESO] = proc[COL_PROCESO].astype("string")
    print(f"\nBase ajustada encontrada: {RUTA_AJUSTADA.name}  ({len(proc):,} filas)")


else:
    print("\nNo hay base ajustada guardada. Se reconstruye con la lógica de V2.\n")

    proc = data.drop_duplicates(subset=[COL_PROCESO]).copy()

    for nombre, ini, fin in INTERVALOS:
        proc[f"{nombre}_bruto"] = (proc[COL[fin]] - proc[COL[ini]]).dt.days

    umbrales, exclusiones = {}, {}

    for nombre, _, _ in INTERVALOS:
        bruto = proc[f"{nombre}_bruto"]

        negativos = bruto < 0
        valores = bruto.where(~negativos) if DESCARTAR_NEGATIVOS else bruto

        umbral = (
            valores.quantile(PERCENTIL_CORTE)
            if PERCENTIL_CORTE is not None and valores.notna().any()
            else np.nan
        )

        sobre_umbral = (
            valores.notna() & (valores > umbral)
            if pd.notna(umbral)
            else pd.Series(False, index=proc.index)
        )

        proc[nombre] = valores.where(~sobre_umbral)

        umbrales[nombre] = umbral
        exclusiones[nombre] = {
            "válidas antes": int(bruto.notna().sum()),
            "negativas": int(negativos.sum()),
            "sobre el p99": int(sobre_umbral.sum()),
            "umbral (días)": round(umbral, 1) if pd.notna(umbral) else np.nan,
            "válidas finales": int(proc[nombre].notna().sum()),
        }

    print("\nLimpieza aplicada (umbral global sobre la base total):\n")
    print(pd.DataFrame(exclusiones).T.to_string())

    proc.to_parquet(RUTA_AJUSTADA, index=False)
    print(f"\nBase ajustada guardada: {RUTA_AJUSTADA}")


if not RUTA_CAPTURAS.exists():
    raise FileNotFoundError(
        f"No se encontró la base de capturas: {RUTA_CAPTURAS}"
    )


capturas_eventos = pd.read_parquet(
    RUTA_CAPTURAS,
    columns=[COL_ID, COL["captura"], "radicado_consulta"],
).copy()

capturas_eventos[COL["captura"]] = pd.to_datetime(
    capturas_eventos[COL["captura"]],
    errors="coerce",
).astype("datetime64[ns]")

capturas_eventos["_id_busqueda"] = (
    capturas_eventos[COL_ID]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

capturas_eventos["_radicado_busqueda"] = (
    capturas_eventos["radicado_consulta"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
    .fillna("")
)

capturas_eventos = capturas_eventos[
    capturas_eventos["_id_busqueda"].notna()
    & capturas_eventos[COL["captura"]].notna()
].copy()

fecha_llave = capturas_eventos[COL["captura"]].dt.strftime("%Y-%m-%d")

capturas_eventos["llave_captura"] = np.where(
    capturas_eventos["_radicado_busqueda"].ne(""),
    (
        "RAD | "
        + capturas_eventos["_id_busqueda"]
        + " | "
        + capturas_eventos["_radicado_busqueda"]
    ),
    (
        "SINRAD | "
        + capturas_eventos["_id_busqueda"]
        + " | "
        + fecha_llave
    ),
)

capturas_eventos = (
    capturas_eventos
    .drop_duplicates(subset=["llave_captura"])
    .sort_values(["_id_busqueda", COL["captura"]])
    .reset_index(drop=True)
)

print(
    f"\nEventos policiales utilizables: {len(capturas_eventos):,}"
)
print(
    f"  con radicado   : "
    f"{int(capturas_eventos['_radicado_busqueda'].ne('').sum()):,}"
)
print(
    f"  sin radicado   : "
    f"{int(capturas_eventos['_radicado_busqueda'].eq('').sum()):,}"
)

print(f"\nRadicados : {proc[COL_PROCESO].nunique():,}")
print(f"Cédulas   : {proc[COL_ID].nunique():,}")

## Paso 4. Una fila por persona y proceso

La base ajustada se queda como esta y aca se expande con todas las
personas capturadas de cada proceso, pegando las duraciones limpias por
radicado. Sin esto, quien figura como segundo o tercer capturado no entra
en la lista de candidatos y el rearresto de los demas queda subestimado.

In [ ]:
data[COL_LLAVE] = (
    data[COL_ID].astype("string").fillna("SIN_CEDULA")
    + " | "
    + data[COL_PROCESO].astype("string")
)

casos = data.drop_duplicates(subset=[COL_LLAVE]).copy()

COLS_LIMPIAS = [COL_PROCESO] + [c for c in VARIABLES if c in proc.columns]

casos = casos.drop(columns=[c for c in VARIABLES if c in casos.columns]).merge(
    proc[COLS_LIMPIAS], on=COL_PROCESO, how="left"
)

print(f"Filas de la base ajustada (una por radicado) : {len(proc):,}")
print(f"Casos persona x proceso                      : {len(casos):,}")
print(f"  filas recuperadas                          : {len(casos) - len(proc):,}")

personas_por_proceso = casos.groupby(COL_PROCESO)[COL_ID].nunique()
print(f"\nProcesos con más de una persona: "
      f"{int((personas_por_proceso > 1).sum()):,} "
      f"({(personas_por_proceso > 1).mean():.1%})")


proc = casos

sin_cedula = int(proc[COL_ID].isna().sum())
proc = proc[proc[COL_ID].notna()].copy()

proc["fecha_evento_radicado"] = (
    proc[COL["captura"]]
    .fillna(proc[COL["legalizacion"]])
    .fillna(proc[COL["imputacion"]])
)

print("\nCobertura de las fechas que definen la ventana y los eventos:\n")
cobertura = pd.Series({
    "FECHA (captura)": proc[COL["captura"]].notna().mean(),
    "FechaLegalizacion": proc[COL["legalizacion"]].notna().mean(),
    "FechaImputacion": proc[COL["imputacion"]].notna().mean(),
    "FechaMedida": proc[COL["medida"]].notna().mean(),
    "FechaBoletaSalida": proc[COL["boleta_salida"]].notna().mean(),
    "fecha_evento_radicado": proc["fecha_evento_radicado"].notna().mean(),
})
print((cobertura * 100).round(1).to_string())

print(f"\nRegistros sin cédula, excluidos: {sin_cedula:,}")
print(f"Registros de trabajo           : {len(proc):,}")

## Pasos 5 a 9. La ventana

Grupo 1 (BoletaSalida == 1): la ventana arranca en la fecha de la boleta.

Grupo 2 (BoletaSalida == 0): no hay fecha de salida, asi que se imputa
sumando a FechaMedida el promedio de dias_legalizacion_boleta_salida. El
promedio no es uno solo para toda la base: se calcula por celda de casos
parecidos y a cada caso se le asigna el de su celda.

La celda es la combinacion mas especifica de juzgado, delito y municipio
que tenga al menos N_MINIMO casos con boleta. Si no llega, se baja un
escalon (se suelta el juzgado, despues el municipio) hasta el promedio
global, que queda como ultimo recurso. Cada caso guarda en nivel_match el
escalon con el que se resolvio y en n_celda_match el tamano de la celda.

### Paso 5a. Las celdas del match

Busca las columnas de juzgado, delito, municipio y departamento por
nombre, las normaliza (tildes, mayusculas, espacios de mas) y arma la
escalera.

In [ ]:
import re
import unicodedata

COLS_MATCH = {
    "juzgado": None,
    "delito": None,
    "municipio": None,
    "departamento": None,
}

CANDIDATOS = {
    "juzgado": ["juzgado", "despacho", "juzgadoconocimiento", "juez"],
    "delito": ["delito", "conducta", "tipopenal", "tipodelito", "descripciondelito"],
    "municipio": ["municipio", "mpio", "ciudad", "municipiohechos"],
    "departamento": ["departamento", "depto", "dpto"],
}

ESCALERA = [
    ("juzgado + delito + municipio", ["juzgado", "delito", "municipio"]),
    ("delito + municipio",           ["delito", "municipio"]),
    ("juzgado + delito",             ["juzgado", "delito"]),
    ("delito + departamento",        ["delito", "departamento"]),
    ("delito",                       ["delito"]),
    ("municipio",                    ["municipio"]),
]

N_MINIMO = 30
SIN_DATO = "SIN_DATO"


def _clave_nombre(texto: str) -> str:
    limpio = unicodedata.normalize("NFKD", str(texto))
    limpio = limpio.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]", "", limpio.lower())


def resolver_columna(df, nombre_dado, candidatos):
    if nombre_dado is not None:
        if nombre_dado not in df.columns:
            raise KeyError(f"La columna '{nombre_dado}' no esta en la base")
        return nombre_dado

    disponibles = {_clave_nombre(c): c for c in df.columns}

    for candidato in candidatos:
        clave = _clave_nombre(candidato)
        if clave in disponibles:
            return disponibles[clave]

    for candidato in candidatos:
        clave = _clave_nombre(candidato)
        for clave_col, original in disponibles.items():
            if clave in clave_col:
                return original

    return None


def _limpiar_valor(valor: str) -> str:
    limpio = unicodedata.normalize("NFKD", str(valor))
    limpio = limpio.encode("ascii", "ignore").decode("ascii").upper()
    limpio = re.sub(r"[^A-Z0-9]+", " ", limpio)
    return limpio.strip()


def normalizar_serie(serie: pd.Series) -> pd.Series:
    mapa = {v: _limpiar_valor(v) for v in serie.dropna().unique()}
    salida = serie.map(mapa).astype("object")
    salida = salida.where(salida.notna() & salida.ne(""), SIN_DATO)
    return salida


COLS_MATCH = {
    rol: resolver_columna(proc, COLS_MATCH.get(rol), candidatos)
    for rol, candidatos in CANDIDATOS.items()
}

print("Columnas de match encontradas:\n")
for rol, columna in COLS_MATCH.items():
    if columna is None:
        print(f"  {rol:<14} -> (no encontrada, ese escalon se salta)")
    else:
        n_niveles = proc[columna].nunique(dropna=True)
        cobertura = proc[columna].notna().mean()
        print(f"  {rol:<14} -> {columna:<28} "
              f"{n_niveles:>6,} categorias   {cobertura:.1%} con dato")


ROLES_OK = {rol for rol, col in COLS_MATCH.items() if col is not None}

if "delito" not in ROLES_OK:
    print("\nOJO: sin columna de delito el match pierde casi todo su sentido. "
          "Revisa los nombres de columna y escribilos a mano en COLS_MATCH.")


for rol in ROLES_OK:
    proc[f"k_{rol}"] = normalizar_serie(proc[COLS_MATCH[rol]])


ESCALERA_ACTIVA, vistos = [], set()
for etiqueta, roles in ESCALERA:
    if not set(roles).issubset(ROLES_OK):
        continue
    firma = tuple(sorted(roles))
    if firma in vistos:
        continue
    vistos.add(firma)
    ESCALERA_ACTIVA.append((etiqueta, roles))


print("\nEscalera activa:")
for i, (etiqueta, _) in enumerate(ESCALERA_ACTIVA, start=1):
    print(f"  {i}. {etiqueta}")
print(f"  {len(ESCALERA_ACTIVA) + 1}. promedio global (ultimo recurso)")
print(f"\nMinimo de casos con boleta por celda: {N_MINIMO}")

In [ ]:
con_boleta = proc[COL_DUMMY_BOLETA].eq(1) & proc[COL["boleta_salida"]].notna()

print(f"Grupo 1 — con boleta de salida : {int(con_boleta.sum()):,} "
      f"({con_boleta.mean():.1%})")
print(f"Grupo 2 — sin boleta de salida : {int((~con_boleta).sum()):,} "
      f"({(~con_boleta).mean():.1%})")


dias_leg_boleta = proc.loc[con_boleta, "dias_legalizacion_boleta_salida"]
promedio_dias = dias_leg_boleta.mean()

print(f"\ndias_legalizacion_a_boleta (valores ya ajustados de V2):")
print(f"  n         : {int(dias_leg_boleta.notna().sum()):,}")
print(f"  promedio  : {promedio_dias:.1f} días   <- respaldo global")
print(f"  mediana   : {dias_leg_boleta.median():.1f} días")
print(f"  máximo    : {dias_leg_boleta.max():.0f} días")

base_celdas = (
    proc.loc[con_boleta & proc["dias_legalizacion_boleta_salida"].notna()]
    .drop_duplicates(subset=[COL_PROCESO])
    .copy()
)

print(f"\nBase de celdas: {len(base_celdas):,} radicados con boleta y duración válida")


def llave_celda(df: pd.DataFrame, roles: list) -> pd.Series:
    serie = df[f"k_{roles[0]}"].astype("object")
    for rol in roles[1:]:
        serie = serie + " | " + df[f"k_{rol}"].astype("object")
    return serie


def celda_completa(df: pd.DataFrame, roles: list) -> pd.Series:
    completa = pd.Series(True, index=df.index)
    for rol in roles:
        completa &= df[f"k_{rol}"].ne(SIN_DATO)
    return completa


dias_estimados = pd.Series(np.nan, index=proc.index)
nivel_match = pd.Series("", index=proc.index, dtype="object")
n_celda_match = pd.Series(np.nan, index=proc.index)

trazas = []

for etiqueta, roles in ESCALERA_ACTIVA:
    pendientes = dias_estimados.isna()
    if not pendientes.any():
        break

    ref = base_celdas.loc[celda_completa(base_celdas, roles)]
    tabla = (
        ref.groupby(llave_celda(ref, roles), sort=False)[
            "dias_legalizacion_boleta_salida"
        ]
        .agg(promedio="mean", n="count")
    )
    tabla = tabla[tabla["n"] >= N_MINIMO]

    objetivo = pendientes & celda_completa(proc, roles)
    if not objetivo.any() or tabla.empty:
        trazas.append({
            "escalón": etiqueta,
            "celdas válidas": len(tabla),
            "casos asignados": 0,
            "promedio asignado": np.nan,
        })
        continue

    llaves = llave_celda(proc.loc[objetivo], roles)
    valores = llaves.map(tabla["promedio"])
    conteos = llaves.map(tabla["n"])
    acertados = valores.notna()
    indice = valores.index[acertados]

    dias_estimados.loc[indice] = valores.loc[indice].to_numpy()
    nivel_match.loc[indice] = etiqueta
    n_celda_match.loc[indice] = conteos.loc[indice].to_numpy()

    trazas.append({
        "escalón": etiqueta,
        "celdas válidas": len(tabla),
        "casos asignados": int(acertados.sum()),
        "promedio asignado": round(float(valores.loc[indice].mean()), 1)
        if acertados.any() else np.nan,
    })


sin_celda = dias_estimados.isna()
dias_estimados.loc[sin_celda] = promedio_dias
nivel_match.loc[sin_celda] = "global"
n_celda_match.loc[sin_celda] = int(dias_leg_boleta.notna().sum())

trazas.append({
    "escalón": "global",
    "celdas válidas": 1,
    "casos asignados": int(sin_celda.sum()),
    "promedio asignado": round(float(promedio_dias), 1),
})

print("\nAsignación por escalón (todas las filas):\n")
print(pd.DataFrame(trazas).to_string(index=False))

proc["dias_salida_estimados"] = dias_estimados
proc["nivel_match"] = nivel_match
proc["n_celda_match"] = n_celda_match

print("\nEscalón usado en el Grupo 2, que es donde el match cambia algo:\n")
reparto = (
    proc.loc[~con_boleta, "nivel_match"]
    .value_counts()
    .rename("casos")
    .to_frame()
)
reparto["%"] = (reparto["casos"] / len(proc.loc[~con_boleta]) * 100).round(1)
print(reparto.to_string())

dispersion = proc.loc[~con_boleta, "dias_salida_estimados"]
print(f"\nDías estimados en el Grupo 2 (antes eran {promedio_dias:.1f} para todos):")
print(f"  promedio  : {dispersion.mean():.1f}")
print(f"  mínimo    : {dispersion.min():.1f}")
print(f"  máximo    : {dispersion.max():.1f}")
print(f"  desv. est.: {dispersion.std():.1f}")

proc["fecha_salida_estimada"] = proc[COL["medida"]] + pd.to_timedelta(
    proc["dias_salida_estimados"], unit="D"
)

proc["fecha_inicio_seguimiento"] = np.where(
    con_boleta, proc[COL["boleta_salida"]], proc["fecha_salida_estimada"]
)
proc["fecha_inicio_seguimiento"] = pd.to_datetime(proc["fecha_inicio_seguimiento"])

proc["fecha_fin_seguimiento"] = proc["fecha_inicio_seguimiento"] + pd.Timedelta(
    days=VENTANA_DIAS
)

proc["origen_ventana"] = np.select(
    [con_boleta, proc["fecha_salida_estimada"].notna()],
    ["boleta de salida", "estimada desde la medida"],
    default="sin ventana",
)

print("\nOrigen de la ventana:")
print(proc["origen_ventana"].value_counts().to_string())

sin_ventana = proc["fecha_inicio_seguimiento"].isna()
print(f"\nCasos sin ventana (sin boleta y sin fecha de medida): "
      f"{int(sin_ventana.sum()):,} ({sin_ventana.mean():.1%})")

## Pasos 10 a 12. Las tres variables

Misma ventana; lo que cambia es la fecha del otro radicado que tiene que
caer adentro: FechaLegalizacion para rearresto, FechaImputacion para
reimputacion y fecha_evento_radicado para caso abierto. En los tres:
misma cedula, radicado distinto y fecha entre inicio y fin incluidos.

In [ ]:
def marcar(columna_fecha: str, nombre_variable: str) -> pd.Series:
    candidatos = proc.loc[
        proc[columna_fecha].notna(), [COL_ID, columna_fecha, COL_PROCESO]
    ].sort_values([COL_ID, columna_fecha])

    por_persona = {
        cedula: (
            grupo[columna_fecha].to_numpy(),
            grupo[COL_PROCESO].to_numpy(),
        )
        for cedula, grupo in candidatos.groupby(COL_ID, sort=False)
    }

    marcas = np.zeros(len(proc), dtype=float)

    for i, (cedula, inicio, fin, radicado) in enumerate(
        zip(proc[COL_ID], proc["fecha_inicio_seguimiento"],
            proc["fecha_fin_seguimiento"], proc[COL_PROCESO])
    ):
        if pd.isna(inicio) or cedula not in por_persona:
            marcas[i] = np.nan if pd.isna(inicio) else 0.0
            continue

        fechas, radicados = por_persona[cedula]

        desde = np.searchsorted(fechas, np.datetime64(inicio), side="left")
        hasta = np.searchsorted(fechas, np.datetime64(fin), side="right")

        marcas[i] = float((radicados[desde:hasta] != radicado).any())

    return pd.Series(marcas, index=proc.index, name=nombre_variable)


def marcar_rearresto_poli(
    columna_fin: str,
    nombre_variable: str,
) -> pd.Series:
    candidatos = capturas_eventos[
        [
            "_id_busqueda",
            COL["captura"],
            "_radicado_busqueda",
            "llave_captura",
        ]
    ].sort_values(
        ["_id_busqueda", COL["captura"]]
    )

    por_persona = {
        cedula: (
            grupo[COL["captura"]].to_numpy(),
            grupo["_radicado_busqueda"].to_numpy(dtype=str),
        )
        for cedula, grupo in candidatos.groupby("_id_busqueda", sort=False)
    }

    cedulas_proc = (
        proc[COL_ID]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    radicados_proc = (
        proc[COL_PROCESO]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .fillna("")
    )

    fechas_actuales = pd.to_datetime(
        proc[COL["captura"]],
        errors="coerce",
    )

    marcas = np.zeros(len(proc), dtype=float)

    for i, (cedula, inicio, fin, radicado_actual, fecha_actual) in enumerate(
        zip(
            cedulas_proc,
            proc["fecha_inicio_seguimiento"],
            proc[columna_fin],
            radicados_proc,
            fechas_actuales,
        )
    ):
        if pd.isna(inicio) or pd.isna(fin):
            marcas[i] = np.nan
            continue

        if pd.isna(cedula) or cedula not in por_persona:
            marcas[i] = 0.0
            continue

        fechas, radicados = por_persona[cedula]

        desde = np.searchsorted(
            fechas,
            np.datetime64(inicio),
            side="left",
        )
        hasta = np.searchsorted(
            fechas,
            np.datetime64(fin),
            side="right",
        )

        fechas_ventana = fechas[desde:hasta]
        radicados_ventana = radicados[desde:hasta]

        if len(fechas_ventana) == 0:
            marcas[i] = 0.0
            continue

        mismo_radicado = (
            (radicados_ventana != "")
            & (radicado_actual != "")
            & (radicados_ventana == radicado_actual)
        )

        misma_captura_sin_radicado = np.zeros(
            len(fechas_ventana),
            dtype=bool,
        )

        if pd.notna(fecha_actual):
            misma_captura_sin_radicado = (
                (radicados_ventana == "")
                & (
                    fechas_ventana
                    == np.datetime64(fecha_actual)
                )
            )

        es_otra_captura = ~(
            mismo_radicado
            | misma_captura_sin_radicado
        )

        marcas[i] = float(es_otra_captura.any())

    return pd.Series(
        marcas,
        index=proc.index,
        name=nombre_variable,
    )


proc["rearresto"] = marcar(COL["legalizacion"], "rearresto")
proc["reimputacion"] = marcar(COL["imputacion"], "reimputacion")
proc["caso_abierto"] = marcar("fecha_evento_radicado", "caso_abierto")
proc["rearresto_poli"] = marcar_rearresto_poli(
    "fecha_fin_seguimiento",
    "rearresto_poli",
)

resumen = pd.DataFrame({
    variable: {
        "casos con ventana": int(proc[variable].notna().sum()),
        "= 1": int(proc[variable].sum()),
        "= 0": int((proc[variable] == 0).sum()),
        "tasa (%)": round(proc[variable].mean() * 100, 1),
    }
    for variable in ["rearresto", "reimputacion", "caso_abierto", "rearresto_poli"]
}).T

print(resumen.to_string())

print("\nCruce entre las tres variables:")
print(
    proc.groupby(["caso_abierto", "rearresto", "reimputacion", "rearresto_poli"], dropna=False)
    .size()
    .rename("casos")
    .to_string()
)

## Paso 13. Guardado

In [ ]:
NUEVAS = [
    "fecha_evento_radicado",
    "dias_salida_estimados",
    "nivel_match",
    "n_celda_match",
    "fecha_salida_estimada",
    "fecha_inicio_seguimiento",
    "fecha_fin_seguimiento",
    "origen_ventana",
    "rearresto",
    "reimputacion",
    "caso_abierto",
    "rearresto_poli",
]

print("Cobertura de las variables creadas:\n")
print((proc[NUEVAS].notna().mean() * 100).round(1).to_string())

proc.to_parquet(RUTA_SALIDA, index=False)

print(f"\nGuardado: {RUTA_SALIDA}")
print(f"{len(proc):,} filas x {len(proc.columns)} columnas")

print("\nMuestra de las variables nuevas:")
print(
    proc.loc[
        proc["caso_abierto"].eq(1),
        [COL_ID, COL_PROCESO, "origen_ventana", "nivel_match",
         "fecha_inicio_seguimiento",
         "fecha_fin_seguimiento", "rearresto", "reimputacion", "caso_abierto"],
    ].head(10).to_string(index=False)
)

In [ ]:
con_boleta = proc[COL_DUMMY_BOLETA].eq(1) & proc[COL["boleta_salida"]].notna()

print(f"Grupo 1 — con boleta de salida : {int(con_boleta.sum()):,} "
      f"({con_boleta.mean():.1%})")
print(f"Grupo 2 — sin boleta de salida : {int((~con_boleta).sum()):,} "
      f"({(~con_boleta).mean():.1%})")


dias_leg_boleta = proc.loc[con_boleta, "dias_legalizacion_boleta_salida"]
promedio_dias = dias_leg_boleta.mean()

print(f"\ndias_legalizacion_a_boleta (valores ya ajustados de V2):")
print(f"  n         : {int(dias_leg_boleta.notna().sum()):,}")
print(f"  promedio  : {promedio_dias:.1f} días   <- respaldo global")
print(f"  mediana   : {dias_leg_boleta.median():.1f} días")
print(f"  máximo    : {dias_leg_boleta.max():.0f} días")

base_celdas = (
    proc.loc[con_boleta & proc["dias_legalizacion_boleta_salida"].notna()]
    .drop_duplicates(subset=[COL_PROCESO])
    .copy()
)

print(f"\nBase de celdas: {len(base_celdas):,} radicados con boleta y duración válida")


def llave_celda(df: pd.DataFrame, roles: list) -> pd.Series:
    serie = df[f"k_{roles[0]}"].astype("object")
    for rol in roles[1:]:
        serie = serie + " | " + df[f"k_{rol}"].astype("object")
    return serie


def celda_completa(df: pd.DataFrame, roles: list) -> pd.Series:
    completa = pd.Series(True, index=df.index)
    for rol in roles:
        completa &= df[f"k_{rol}"].ne(SIN_DATO)
    return completa


dias_estimados = pd.Series(np.nan, index=proc.index)
nivel_match = pd.Series("", index=proc.index, dtype="object")
n_celda_match = pd.Series(np.nan, index=proc.index)

trazas = []

for etiqueta, roles in ESCALERA_ACTIVA:
    pendientes = dias_estimados.isna()
    if not pendientes.any():
        break

    ref = base_celdas.loc[celda_completa(base_celdas, roles)]
    tabla = (
        ref.groupby(llave_celda(ref, roles), sort=False)[
            "dias_legalizacion_boleta_salida"
        ]
        .agg(promedio="mean", n="count")
    )
    tabla = tabla[tabla["n"] >= N_MINIMO]

    objetivo = pendientes & celda_completa(proc, roles)
    if not objetivo.any() or tabla.empty:
        trazas.append({
            "escalón": etiqueta,
            "celdas válidas": len(tabla),
            "casos asignados": 0,
            "promedio asignado": np.nan,
        })
        continue

    llaves = llave_celda(proc.loc[objetivo], roles)
    valores = llaves.map(tabla["promedio"])
    conteos = llaves.map(tabla["n"])
    acertados = valores.notna()
    indice = valores.index[acertados]

    dias_estimados.loc[indice] = valores.loc[indice].to_numpy()
    nivel_match.loc[indice] = etiqueta
    n_celda_match.loc[indice] = conteos.loc[indice].to_numpy()

    trazas.append({
        "escalón": etiqueta,
        "celdas válidas": len(tabla),
        "casos asignados": int(acertados.sum()),
        "promedio asignado": round(float(valores.loc[indice].mean()), 1)
        if acertados.any() else np.nan,
    })


sin_celda = dias_estimados.isna()
dias_estimados.loc[sin_celda] = promedio_dias
nivel_match.loc[sin_celda] = "global"
n_celda_match.loc[sin_celda] = int(dias_leg_boleta.notna().sum())

trazas.append({
    "escalón": "global",
    "celdas válidas": 1,
    "casos asignados": int(sin_celda.sum()),
    "promedio asignado": round(float(promedio_dias), 1),
})

print("\nAsignación por escalón (todas las filas):\n")
print(pd.DataFrame(trazas).to_string(index=False))

proc["dias_salida_estimados_2"] = dias_estimados
proc["nivel_match"] = nivel_match
proc["n_celda_match"] = n_celda_match

print("\nEscalón usado en el Grupo 2, que es donde el match cambia algo:\n")
reparto = (
    proc.loc[~con_boleta, "nivel_match"]
    .value_counts()
    .rename("casos")
    .to_frame()
)
reparto["%"] = (reparto["casos"] / len(proc.loc[~con_boleta]) * 100).round(1)
print(reparto.to_string())

dispersion = proc.loc[~con_boleta, "dias_salida_estimados_2"]
print(f"\nDías estimados en el Grupo 2 (antes eran {promedio_dias:.1f} para todos):")
print(f"  promedio  : {dispersion.mean():.1f}")
print(f"  mínimo    : {dispersion.min():.1f}")
print(f"  máximo    : {dispersion.max():.1f}")
print(f"  desv. est.: {dispersion.std():.1f}")

proc["fecha_salida_estimada_2"] = proc[COL["medida"]] + pd.to_timedelta(
    proc["dias_salida_estimados_2"], unit="D"
)

proc["fecha_inicio_seguimiento"] = np.where(
    con_boleta, proc[COL["boleta_salida"]], proc["fecha_salida_estimada"]
)
proc["fecha_inicio_seguimiento"] = pd.to_datetime(proc["fecha_inicio_seguimiento"])

proc["fecha_fin_seguimiento_2"] = proc["fecha_inicio_seguimiento"] + pd.Timedelta(
    days=730
)

proc["origen_ventana"] = np.select(
    [con_boleta, proc["fecha_salida_estimada_2"].notna()],
    ["boleta de salida", "estimada desde la medida"],
    default="sin ventana",
)


def marcar(columna_fecha: str, nombre_variable: str) -> pd.Series:
    candidatos = proc.loc[
        proc[columna_fecha].notna(), [COL_ID, columna_fecha, COL_PROCESO]
    ].sort_values([COL_ID, columna_fecha])

    por_persona = {
        cedula: (
            grupo[columna_fecha].to_numpy(),
            grupo[COL_PROCESO].to_numpy(),
        )
        for cedula, grupo in candidatos.groupby(COL_ID, sort=False)
    }

    marcas = np.zeros(len(proc), dtype=float)

    for i, (cedula, inicio, fin, radicado) in enumerate(
        zip(proc[COL_ID], proc["fecha_inicio_seguimiento"],
            proc["fecha_fin_seguimiento_2"], proc[COL_PROCESO])
    ):
        if pd.isna(inicio) or cedula not in por_persona:
            marcas[i] = np.nan if pd.isna(inicio) else 0.0
            continue

        fechas, radicados = por_persona[cedula]

        desde = np.searchsorted(fechas, np.datetime64(inicio), side="left")
        hasta = np.searchsorted(fechas, np.datetime64(fin), side="right")

        marcas[i] = float((radicados[desde:hasta] != radicado).any())

    return pd.Series(marcas, index=proc.index, name=nombre_variable)


proc["rearresto_2"] = marcar(COL["legalizacion"], "rearresto_2")
proc["reimputacion_2"] = marcar(COL["imputacion"], "reimputacion_2")
proc["caso_abierto_2"] = marcar("fecha_evento_radicado", "caso_abierto_2")
proc["rearresto_poli_2"] = marcar_rearresto_poli(
    "fecha_fin_seguimiento_2",
    "rearresto_poli_2",
)

resumen = pd.DataFrame({
    variable: {
        "casos con ventana": int(proc[variable].notna().sum()),
        "= 1": int(proc[variable].sum()),
        "= 0": int((proc[variable] == 0).sum()),
        "tasa (%)": round(proc[variable].mean() * 100, 1),
    }
    for variable in ["rearresto_2", "reimputacion_2", "caso_abierto_2", "rearresto_poli_2"]
}).T

print(resumen.to_string())

print("\nCruce entre las tres variables:")
print(
    proc.groupby(["caso_abierto_2", "rearresto_2", "reimputacion_2", "rearresto_poli_2"], dropna=False)
    .size()
    .rename("casos")
    .to_string()
)

In [ ]:
NUEVAS = [
    "fecha_evento_radicado",
    "dias_salida_estimados",
    "nivel_match",
    "n_celda_match",
    "fecha_salida_estimada",
    "fecha_inicio_seguimiento",
    "fecha_fin_seguimiento",
    "origen_ventana",
    "rearresto",
    "reimputacion",
    "caso_abierto",
    "rearresto_poli",
    "rearresto_2",
    "reimputacion_2",
    "caso_abierto_2",
    "rearresto_poli_2"


]

print("Cobertura de las variables creadas:\n")
print((proc[NUEVAS].notna().mean() * 100).round(1).to_string())

proc.to_parquet(RUTA_SALIDA, index=False)

print(f"\nGuardado: {RUTA_SALIDA}")
print(f"{len(proc):,} filas x {len(proc.columns)} columnas")

print("\nMuestra de las variables nuevas:")
print(
    proc.loc[
        proc["caso_abierto"].eq(1),
        [COL_ID, COL_PROCESO, "origen_ventana", "nivel_match",
         "fecha_inicio_seguimiento",
         "fecha_fin_seguimiento", "rearresto", "reimputacion", "caso_abierto"],
    ].head(10).to_string(index=False)
)

In [ ]:
PARES = [
    ("rearresto", "rearresto_2"),
    ("reimputacion", "reimputacion_2"),
    ("caso_abierto", "caso_abierto_2"),
    ("rearresto_poli", "rearresto_poli_2"),
]

VARIABLES_CONTEO = [v for par in PARES for v in par if v in proc.columns]

conteo = pd.DataFrame([
    {
        "variable": v,
        "con ventana": int(proc[v].notna().sum()),
        "= 1": int(proc[v].eq(1).sum()),
        "= 0": int(proc[v].eq(0).sum()),
        "sin ventana (NaN)": int(proc[v].isna().sum()),
        "tasa (%)": round(proc[v].mean() * 100, 2),
    }
    for v in VARIABLES_CONTEO
])

print("Conteo de unos por variable:\n")
print(conteo.to_string(index=False))

print("\n\nCambios entre la versión 1 y la 2:\n")
for a, b in PARES:
    if a not in proc.columns or b not in proc.columns:
        continue
    tabla = pd.crosstab(proc[a], proc[b], dropna=False)
    solo_1 = int(proc[a].eq(1).sum() - proc[[a, b]].eq(1).all(axis=1).sum())
    solo_2 = int(proc[b].eq(1).sum() - proc[[a, b]].eq(1).all(axis=1).sum())
    print(f"{a}  vs  {b}")
    print(tabla.to_string())
    print(f"  marcados solo en {a}: {solo_1:,}")
    print(f"  marcados solo en {b}: {solo_2:,}")
    print(f"  marcados en ambas   : {int(proc[[a, b]].eq(1).all(axis=1).sum()):,}\n")

### Dos cosas para tener presentes

1. La limpieza sigue siendo por radicado y el seguimiento por persona. Si
   la limpieza se hiciera sobre la base expandida, un proceso con tres
   capturados aportaria su duracion tres veces al percentil y los umbrales
   se moverian.
2. Los casos sin ventana quedan en NaN y no en 0. Son los que no tienen ni
   boleta de salida ni fecha de medida: no hay desde donde contar los 365
   dias, y marcarlos como 0 diria que no hubo rearresto cuando en realidad
   no se pudo mirar.